In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, time, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES DESTINO

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="GCSPE"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = 'USR_GSIT_GCHIARELLA'
pw = 'eBLfq$GQ6ZJ83P'
hostname='172.20.0.185'
service_name="dbsas"
port = 1521

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

In [4]:

query="""select 
CASE sd.txt_ipusuario_crea WHEN '1.1.1.1' THEN 'VIVA' ELSE 'SAS' END as ORIGEN,
'MATERNIDAD' as TRANSACCION,
sd.COD_EDOC_BENEFICIARIO AS TIPO_DOC,
TO_CHAR(sd.NUM_NDOC_BENEFICIARIO) AS NUM_DOC,
sd.NUM_RUC_EMPLEADOR AS RUC,
sd.num_expediente,
sd.fec_usuario_crea,
sd.COD_USUARIO_CREA,
'0' AS NUM_DEPENDENCIA
from usrcsa.ssutdetmat sd
where sd.NUM_EXPEDIENTE is not null
--and to_char(fec_usuario_crea,'yyyy') IN ('2024') 

UNION ALL
select 
CASE sd.txt_ipusuario_crea WHEN '1.1.1.1' THEN 'VIVA' ELSE 'SAS' END as ORIGEN,
'INCAPACIDAD' as TRANSACCION,
sd.COD_EDOC_BENEFICIARIO AS TIPO_DOC,
sd.NUM_NDOC_BENEFICIARIO AS NUM_DOC,
sd.NUM_RUC_EMPLEADOR AS RUC,
sd.num_expediente,
sd.fec_usuario_crea,
sd.COD_USUARIO_CREA,
'0' AS NUM_DEPENDENCIA
from usrcsa.ssutdetinc sd 
where sd.NUM_EXPEDIENTE is not null 
--and to_char(sd.fec_usuario_crea,'yyyy') in ('2024') 

UNION ALL
select
CASE t.txt_ipusuario_crea WHEN '1.1.1.1' THEN 'VIVA' ELSE 'SAS' END as ORIGEN,
'LACTANCIA' as TRANSACCION,
t.COD_EDOC_TITULAR AS TIPO_DOC,
t.NUM_DOC_TITULAR AS NUM_DOC,
t.COD_EMPLEADOR AS RUC,
t.cod_expediente_nit,
t.fec_usuario_crea,
t.COD_USUARIO_CREA,
'0' AS NUM_DEPENDENCIA
from usrcsa.ssutsolici t
      LEFT JOIN usrcsa.SSUTDETLAC D
        ON T.IDE_NUM_SOLICITU = D.COD_IDE_SOLICITUD
      left join usrcsa.ssutcalifi c
        on t.ide_num_solicitu = c.cod_solici_replac
      left join usrcsa.ssumresult r
        on c.cod_result_detalle = r.ide_num_result
where 
--to_char(t.fec_usuario_crea,'yyyy') in ('2024') and 
cod_usuario_act <> 'LOFICIO'
AND t.cod_etipo_subsidio in ('LC','LT')

UNION ALL
select 
CASE t.txt_ipusuario_crea WHEN '1.1.1.1' THEN 'VIVA' ELSE 'SAS' END as ORIGEN,
'SEPELIO' as TRANSACCION,  
t.COD_EDOC_TITULAR AS TIPO_DOC,
t.NUM_DOC_TITULAR AS NUM_DOC,
t.COD_EMPLEADOR AS RUC,
t.cod_expediente_nit,
t.fec_usuario_crea,
t.COD_USUARIO_CREA,
'0' AS NUM_DEPENDENCIA
from usrcsa.ssutsolici t
      LEFT JOIN usrcsa.SSUTDETSEP D
        ON T.IDE_NUM_SOLICITU = D.COD_IDE_SOLICITUD
      left join usrcsa.ssutcalifi c
        on t.ide_num_solicitu = c.cod_solici_replac
      left join usrcsa.ssumresult r
        on c.cod_result_detalle = r.ide_num_result
where 
--to_char(t.fec_usuario_crea,'yyyy')in ('2024') 
--AND 
t.cod_etipo_subsidio in ('SE') and t.cod_eestado_subsidio in ('1','2')

UNION ALL
---TRANSACCIONES DE PRESTACIONES ECONOMICAS REGISTRADAS EN SIA (PHP)
select 
'PHP' as ORIGEN,
CASE SSU.ASECTSU
  WHEN 'AC' THEN 'INCAPACIDAD'
  WHEN 'EN' THEN 'INCAPACIDAD'
  WHEN 'EP' THEN 'INCAPACIDAD'  
  WHEN 'AT' THEN 'INCAPACIDAD'
  WHEN 'MA' THEN 'MATERNIDAD'
  WHEN 'MD' THEN 'MATERNIDAD'  
  WHEN 'SE' THEN 'SEPELIO'
  WHEN 'LC' THEN 'LACTANCIA'
  WHEN 'LT' THEN 'LACTANCIA'
   END AS TRANSACCION,
SSU.ASETDOC AS TIPO_DOC,
SSU.ASECLEL AS NUM_DOC,
'0' AS RUC,
SSU.ASECEXP AS NUM_EXPEDIENTE,
SSU.ASEFPRE AS fec_usuario_crea,
'0' AS COD_USUARIO_CREA,
'0' AS NUM_DEPENDENCIA
 from sia.ssumaset@lnk_sia_sas.essalud SSU
where nvl(asetcnv,1)>0
--AND TO_CHAR(ASEFPRE,'YYYY')='2024'

UNION ALL
select 
  'VIVA' as ORIGEN,
  'CANJE CEVIT' AS TRANSACCION,
  TO_CHAR(sdc.COD_EDOC_BENEFICIARIO) AS TIPO_DOC,
  TO_CHAR(sdc.NUM_NDOC_BENEFICIARIO) AS NUM_DOC,
  TO_CHAR(sdc.NUM_RUC_EMPLEADOR) AS RUC,
  CASE SSU.FLG_ESTADO_PRO
    WHEN 0 THEN 'PENDIENTE'
    WHEN 4 THEN 'ENEXEMUNMIT'
    WHEN 5 THEN 'OBSERVADO'
    WHEN 6 THEN 'EN EVALUACION SIGI'
  END AS NUM_EXPEDIENTE,
  SSU.FEC_USUARIO_CREA,
  ssu.NUM_DOC_USUARIO AS COD_USUARIO_CREA,
  od.codigo_dependencia as NUM_DEPENDENCIA
--od.dependencia as DESCRIPCION
from 
  ospe_virtual.ssutsolici_ov ssu
  inner join ospe_virtual.ssutdetcanjecitt_ov sdc on ssu.ide_num_solicitud_ov = sdc.ide_num_solicitud_ov
  left join app_ospe_virtual.ospedependencia@lnk_dbsas_ospev od on ssu.cod_ospe= od.codigo_ospe
where 
  ssu.cod_etipo_subsidio in ('CC') 
"""

query2 = 'SELECT * FROM USRCSA.V_OSPEDEPENDENCIA vo'

base1 = pd.read_sql_query(query, con=connection0)
base2 = pd.read_sql_query(query2, con=connection0)

connection0.close()
engine0.dispose()

base1.to_sql(name=f'TRANSACCIONES_OSPES_PRESTACIONES_ECONOMICAS', con=engine1, if_exists='replace', index=False)
base2.to_sql(name=f'MAESTRO_OSPES', con=engine1, if_exists='replace', index=False)

35